# CISB5123 Text Analytics
## Project Part 1: Data Collection and Preprocessing

### Project Title
Sentiment Analysis of Guest Reviews for Jabal Omar Marriott Hotel, Makkah Using Tripadvisor Data

### Group Members
- ALSAKKAF MUSAAB (SW01083207)
- BAWZIR AHMED ALI AHMED (SW01084040)
- ALHATHAL NAIF MASHAAN D (SW01084553)
- HASHEM MUSAAB (SW01084071)
- Mousab Mohammed Elhassan Adam (SW01084034)
- ZRIYAB ABUBAKER MOSTAFA (SW01082419)

In [8]:
!pip install selenium webdriver-manager pandas

In [9]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [10]:
base_url = "https://www.tripadvisor.com.my/Hotel_Review-g293993-d8142637-Reviews-or{}-Jabal_Omar_Marriott_Hotel_Makkah-Mecca_Makkah_Province.html"

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

print("Browser started.")

Browser started.


In [14]:
def extract_reviews_from_current_page():
    reviews = []

    review_headers = driver.find_elements(By.XPATH, "//*[contains(text(),'wrote a review')]")
    print("Review headers found:", len(review_headers))

    for header in review_headers:
        try:
            header_text = header.text.strip()
            if " wrote a review " not in header_text:
                continue

            reviewer_name, review_date = header_text.split(" wrote a review ", 1)

            # get a bigger parent container that holds the whole review card
            container_text = driver.execute_script("""
                let el = arguments[0];
                let node = el;
                for (let i = 0; i < 8; i++) {
                    if (!node) break;
                    if (node.innerText && node.innerText.length > 200) {
                        return node.innerText;
                    }
                    node = node.parentElement;
                }
                return el.innerText || "";
            """, header)

            lines = [line.strip() for line in container_text.split("\n") if line.strip()]

            # debug first card if needed
            # print(lines)

            # find header line position
            try:
                start_idx = lines.index(header_text)
            except ValueError:
                # sometimes there are extra spaces
                start_idx = -1
                for idx, line in enumerate(lines):
                    if " wrote a review " in line and reviewer_name in line:
                        start_idx = idx
                        break

            if start_idx == -1:
                continue

            # take only lines after the header
            remaining = lines[start_idx + 1:]

            cleaned = []
            for line in remaining:
                low = line.lower()

                # stop when footer/extra stuff starts
                if line.startswith("Date of stay:"):
                    break
                if line.startswith("Trip type:"):
                    break
                if line.startswith("This review is the subjective opinion"):
                    break
                if line.startswith("Response from"):
                    break
                if line.startswith("Dear"):
                    break
                if line.startswith("Visit hotel website"):
                    break
                if line.startswith("Check availability"):
                    break

                # skip metadata lines
                if "contribution" in low:
                    continue
                if "helpful vote" in low:
                    continue
                if "of 5 bubbles" in low:
                    continue
                if low == "read more":
                    continue

                cleaned.append(line)

            if not cleaned:
                continue

            # first meaningful line is usually title, rest is review text
            if len(cleaned) >= 2:
                review_content = " ".join(cleaned[1:]).strip()
            else:
                review_content = cleaned[0].strip()

            review_content = " ".join(review_content.split())

            if len(review_content) >= 30:
                reviews.append({
                    "reviewer_name": reviewer_name.strip(),
                    "review_date": review_date.strip(),
                    "review_content": review_content
                })

        except Exception as e:
            print("Parse error:", e)

    return reviews

## Step 1: Data Collection

In this step, guest reviews are collected from Tripadvisor for Jabal Omar Marriott Hotel, Makkah.

The extracted fields are:
- reviewer name
- review date
- review content

The collected data is stored in CSV format for preprocessing and analysis.

In [15]:
all_reviews = []

for page in range(50):
    offset = page * 10
    url = base_url.format(offset)

    print(f"\nScraping page {page + 1} | offset={offset}")
    print("Opening:", url)

    driver.get(url)
    time.sleep(8)

    # scroll until the text "wrote a review" appears
    found_reviews = False
    for _ in range(20):
        body_text = driver.find_element(By.TAG_NAME, "body").text

        if "wrote a review" in body_text:
            found_reviews = True
            print("Found review text.")
            break

        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(2)

    if not found_reviews:
        print("No review text found on this page.")
        continue

    page_reviews = extract_reviews_from_current_page()
    print("Reviews extracted from this page:", len(page_reviews))

    all_reviews.extend(page_reviews)

    temp_df = pd.DataFrame(all_reviews).drop_duplicates()
    temp_df.to_csv("tripadvisor_jabal_omar_reviews_progress.csv", index=False, encoding="utf-8-sig")

    time.sleep(2)

print("\nFinished scraping all pages.")


Scraping page 1 | offset=0
Opening: https://www.tripadvisor.com.my/Hotel_Review-g293993-d8142637-Reviews-or0-Jabal_Omar_Marriott_Hotel_Makkah-Mecca_Makkah_Province.html
Found review text.
Review headers found: 10
Reviews extracted from this page: 10

Scraping page 2 | offset=10
Opening: https://www.tripadvisor.com.my/Hotel_Review-g293993-d8142637-Reviews-or10-Jabal_Omar_Marriott_Hotel_Makkah-Mecca_Makkah_Province.html
Found review text.
Review headers found: 10
Reviews extracted from this page: 10

Scraping page 3 | offset=20
Opening: https://www.tripadvisor.com.my/Hotel_Review-g293993-d8142637-Reviews-or20-Jabal_Omar_Marriott_Hotel_Makkah-Mecca_Makkah_Province.html
Found review text.
Review headers found: 10
Reviews extracted from this page: 10

Scraping page 4 | offset=30
Opening: https://www.tripadvisor.com.my/Hotel_Review-g293993-d8142637-Reviews-or30-Jabal_Omar_Marriott_Hotel_Makkah-Mecca_Makkah_Province.html
Found review text.
Review headers found: 10
Reviews extracted from this

## Step 1.1: Dataset Cleaning

After scraping, the dataset is checked for:
- duplicate rows
- missing values
- very short review content

This ensures that only valid review records are used in the preprocessing stage.

In [18]:
# Create dataframe from scraped reviews
df = pd.DataFrame(all_reviews)

# Remove exact duplicates
df = df.drop_duplicates()

# Remove rows with missing review content
df = df.dropna(subset=["review_content"])

# Remove rows where review content is too short
df = df[df["review_content"].str.len() >= 20]

# Reset index
df = df.reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head(10)

Dataset shape: (497, 3)


,reviewer_name,review_date,review_content
0,Companion54966611147,5 Mar,"Honest, Mohamed Kamal is the hotel's best staf..."
1,Wael A,4 Mar,Good treatment from brother Samar from laundry
2,Dorsaf A,4 Mar,Very good Service Mohammed Kamel was very helpful
3,مشاري ح,2 Mar,A special thanks to Mr. Anouar from the laundr...
4,Jrid M,1 Mar,Unfortunately the dining experience is very po...
5,Meera A,Feb 2026,The service is top-notch and the staff of Moha...
6,Khaled A,Feb 2026,Mohammed Kamal from Laundary services is doing...
7,Abu M,Feb 2026,I enjoyed Mohamed Kamal service from laundry
8,Shazia L,Feb 2026,Excellent experience! Check in staff gave us a...
9,hamad a,Feb 2026,"Mohammed kamal from room service was amazing, ..."


In [19]:
df.to_csv("tripadvisor_jabal_omar_reviews.csv", index=False, encoding="utf-8-sig")
print("Raw dataset saved as tripadvisor_jabal_omar_reviews.csv")

Raw dataset saved as tripadvisor_jabal_omar_reviews.csv


## Step 2: Text Preprocessing

The collected review text is preprocessed to make it suitable for text analytics.

The preprocessing steps include:
- lowercasing
- URL removal
- punctuation removal
- number removal
- extra space removal
- stopword removal
- lemmatization
- tokenization

In [27]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [28]:
# Convert to lowercase
def convert_to_lowercase(text):
    return str(text).lower()

df["lowercased"] = df["review_content"].apply(convert_to_lowercase)
df[["review_content", "lowercased"]].head()

,review_content,lowercased
0,"Honest, Mohamed Kamal is the hotel's best staf...","honest, mohamed kamal is the hotel's best staf..."
1,Good treatment from brother Samar from laundry,good treatment from brother samar from laundry
2,Very good Service Mohammed Kamel was very helpful,very good service mohammed kamel was very helpful
3,A special thanks to Mr. Anouar from the laundr...,a special thanks to mr. anouar from the laundr...
4,Unfortunately the dining experience is very po...,unfortunately the dining experience is very po...


In [29]:
# Remove URLs
def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

df["urls_removed"] = df["lowercased"].apply(remove_urls)
df[["lowercased", "urls_removed"]].head()

,lowercased,urls_removed
0,"honest, mohamed kamal is the hotel's best staf...","honest, mohamed kamal is the hotel's best staf..."
1,good treatment from brother samar from laundry,good treatment from brother samar from laundry
2,very good service mohammed kamel was very helpful,very good service mohammed kamel was very helpful
3,a special thanks to mr. anouar from the laundr...,a special thanks to mr. anouar from the laundr...
4,unfortunately the dining experience is very po...,unfortunately the dining experience is very po...


In [30]:
# Remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df["punctuation_removed"] = df["urls_removed"].apply(remove_punctuation)
df[["urls_removed", "punctuation_removed"]].head()

,urls_removed,punctuation_removed
0,"honest, mohamed kamal is the hotel's best staf...",honest mohamed kamal is the hotels best staff ...
1,good treatment from brother samar from laundry,good treatment from brother samar from laundry
2,very good service mohammed kamel was very helpful,very good service mohammed kamel was very helpful
3,a special thanks to mr. anouar from the laundr...,a special thanks to mr anouar from the laundry...
4,unfortunately the dining experience is very po...,unfortunately the dining experience is very po...


In [31]:
# Remove numbers
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

df["numbers_removed"] = df["punctuation_removed"].apply(remove_numbers)
df[["punctuation_removed", "numbers_removed"]].head()

,punctuation_removed,numbers_removed
0,honest mohamed kamal is the hotels best staff ...,honest mohamed kamal is the hotels best staff ...
1,good treatment from brother samar from laundry,good treatment from brother samar from laundry
2,very good service mohammed kamel was very helpful,very good service mohammed kamel was very helpful
3,a special thanks to mr anouar from the laundry...,a special thanks to mr anouar from the laundry...
4,unfortunately the dining experience is very po...,unfortunately the dining experience is very po...


In [32]:
# Remove extra spaces
def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

df["extra_spaces_removed"] = df["numbers_removed"].apply(remove_extra_spaces)
df[["numbers_removed", "extra_spaces_removed"]].head()

,numbers_removed,extra_spaces_removed
0,honest mohamed kamal is the hotels best staff ...,honest mohamed kamal is the hotels best staff ...
1,good treatment from brother samar from laundry,good treatment from brother samar from laundry
2,very good service mohammed kamel was very helpful,very good service mohammed kamel was very helpful
3,a special thanks to mr anouar from the laundry...,a special thanks to mr anouar from the laundry...
4,unfortunately the dining experience is very po...,unfortunately the dining experience is very po...


In [33]:
# Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return " ".join(filtered_words)

df["stopwords_removed"] = df["extra_spaces_removed"].apply(remove_stopwords)
df[["extra_spaces_removed", "stopwords_removed"]].head()

,extra_spaces_removed,stopwords_removed
0,honest mohamed kamal is the hotels best staff ...,honest mohamed kamal hotels best staff quick w...
1,good treatment from brother samar from laundry,good treatment brother samar laundry
2,very good service mohammed kamel was very helpful,good service mohammed kamel helpful
3,a special thanks to mr anouar from the laundry...,special thanks mr anouar laundry department de...
4,unfortunately the dining experience is very po...,unfortunately dining experience poor terms cho...


In [37]:
# POS tagging helper for lemmatization
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()

In [35]:
# Lemmatization
def lemmatize_text(text):
    words = word_tokenize(text)
    pos_tags = pos_tag(words)
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    return " ".join(lemmatized_words)

df["lemmatized"] = df["stopwords_removed"].apply(lemmatize_text)
df[["stopwords_removed", "lemmatized"]].head()

,stopwords_removed,lemmatized
0,honest mohamed kamal hotels best staff quick w...,honest mohamed kamal hotel best staff quick we...
1,good treatment brother samar laundry,good treatment brother samar laundry
2,good service mohammed kamel helpful,good service mohammed kamel helpful
3,special thanks mr anouar laundry department de...,special thanks mr anouar laundry department de...
4,unfortunately dining experience poor terms cho...,unfortunately din experience poor term choice ...


In [36]:
# Tokenization
def tokenize_text(text):
    return word_tokenize(text)

df["tokenized"] = df["lemmatized"].apply(tokenize_text)
df[["lemmatized", "tokenized"]].head()

,lemmatized,tokenized
0,honest mohamed kamal hotel best staff quick we...,"[honest, mohamed, kamal, hotel, best, staff, q..."
1,good treatment brother samar laundry,"[good, treatment, brother, samar, laundry]"
2,good service mohammed kamel helpful,"[good, service, mohammed, kamel, helpful]"
3,special thanks mr anouar laundry department de...,"[special, thanks, mr, anouar, laundry, departm..."
4,unfortunately din experience poor term choice ...,"[unfortunately, din, experience, poor, term, c..."


In [38]:
df.to_csv("Processed_Jabal_Omar_Reviews.csv", index=False, encoding="utf-8-sig")
print("Processed dataset saved as Processed_Jabal_Omar_Reviews.csv")

Processed dataset saved as Processed_Jabal_Omar_Reviews.csv


## Conclusion

The Tripadvisor review data for Jabal Omar Marriott Hotel, Makkah was collected and cleaned successfully.

The review text was then preprocessed using standard text preprocessing techniques. The processed dataset is now ready for sentiment analysis and further text analytics tasks in the next part of the project.